In [6]:
# Library imports
import os
import pandas as pd
import datetime

# File paths and names mapping (for better readability in output)
file_mapping = {
    "adv_mt": {
        "path": "Advancer MT_ServiceWatch Log Parameters - 26th August 24.xlsx",
        "sheet": "AMT ServiceWatch Log Parameters",
        "display_name": "AMT"
    },
    "adv_st": {
        "path": "Advancer ST_REQ_5221_ServiceWatch Log Parameters - NAVI ST - 14th Oct 2024.xlsx",
        "sheet": "ServiceWatch Log Parameters",
        "display_name": "Navigator"
    },
    "APU": {
        "path": "APU TriPac_ServiceWatch Log Parameters_G3_PL5_09th June 25 .xlsx",
        "sheet": "APU ServiceWatch Log Params",
        "display_name": "APU"
    },
    "DEET_MT": {
        "path": "DEET_MT_ServiceWatch Version 1.5.xlsx",
        "sheet": "ServiceWatch Specification",
        "display_name": "DMT"
    },
    "DEET_ST": {
        "path": "DEET_ST_Synergy_SW_Parameters_Rev5.2 REQ 552-update.xlsx",
        "sheet": "Sheet2",
        "display_name": "DEET"
    },
    "Nebula": {
        "path": "Nebula_Galaxy_sw_requirements_v7.xlsx",
        "sheet": "ServiceWatch Log Parameters",
        "display_name": "Nebula-Galaxy"
    },
    "RB": {
        "path": "Railblazer_Service watch data logger- Advancer-sDRC - 29th September 23.xlsx",
        "sheet": "ServiceWatch Log Parameters",
        "display_name": "RB"
    }
}

# Dictionary to store detailed SPN information with characteristics
detailed_SPN_dict = {}

# Process each file and capture detailed characteristics
for key, info in file_mapping.items():
    try:
        # Read the Excel file without converting numbers
        df = pd.read_excel(
            info["path"], 
            sheet_name=info["sheet"],
            dtype=str  # Read all columns as strings to preserve formatting
        )
        
        # Find the SPN column
        spn_column = None
        for col in ['SPN', 'Parameter ID', 'Parameter ID (new)']:
            if col in df.columns:
                spn_column = col
                break
                
        if spn_column is None:
            print(f"Warning: No SPN column found in {info['display_name']}")
            continue

        # Clean the DataFrame - remove completely empty rows
        df = df.dropna(how='all')

        # Process each row to capture characteristics
        for _, row in df.iterrows():
            spn = row[spn_column]
            
            # Enhanced SPN cleaning
            try:
                # Convert to string and clean
                spn = str(spn).strip().lower()
                
                # Remove any non-alphanumeric characters except 'x'
                spn = ''.join(c for c in spn if c.isalnum() or c == 'x')
                
                # Handle different formats of SPN
                if spn.startswith('0x'):
                    # Handle hexadecimal format
                    spn = f"0x{int(spn[2:], 16):04x}"
                elif spn.isdigit():
                    # Handle decimal format
                    spn = f"0x{int(spn):04x}"
                elif len(spn) <= 4 and all(c in '0123456789abcdef' for c in spn):
                    # Handle short hex format without '0x'
                    spn = f"0x{int(spn, 16):04x}"
                else:
                    raise ValueError(f"Invalid SPN format: {spn}")
                    
            except (ValueError, TypeError) as e:
                print(f"Warning: Invalid SPN format in {info['display_name']}: {row[spn_column]} - {str(e)}")
                continue
                
            if spn not in detailed_SPN_dict:
                detailed_SPN_dict[spn] = []
                
            # Enhanced field cleaning function
            def clean_field(value):
                if pd.isna(value):
                    return ''
                    
                # Convert to string and clean
                value = str(value).strip()
                
                # Check for empty or special values
                if value.lower() in ['', 'nan', 'none', '#n/a', 'n/a']:
                    return ''
                    
                # Remove excessive whitespace and normalize
                value = ' '.join(value.split()).lower()
                return value
                
            # Create a characteristics dictionary with cleaned values
            characteristics = {
                'Platform': info['display_name'],
                'Parameter_Name': clean_field(row.get('Parameter Name (as read in SW data log)')),
                'Display_Type': clean_field(row.get('Display Type(s)')),
                'Log_Type': clean_field(row.get('Log Type')),
                'Event_Codes': clean_field(row.get('Event log when alarm code(s) set')),
                'Time_Dependency': clean_field(row.get('TIME ?')),
                'Event_Dependency': clean_field(row.get('Event?')),
                'Additional_Dependencies': clean_field(row.get('Dependency')),
                'Auto_Log_Power': clean_field(row.get('Auto Log at Power ON?')),
                'Auto_Log_Time': clean_field(row.get('Auto Log at 12:05?')),
                'Event_Log_Changes': clean_field(row.get('Event Log When Parameter Changes?'))
            }
            
            # More lenient check for empty characteristics
            non_empty_chars = sum(1 for k, v in characteristics.items() 
                                if k != 'Platform' and v.strip() != '')
            
            if non_empty_chars == 0:
                print(f"Warning: All characteristics empty for SPN {spn} in {info['display_name']}")
                continue
            
            # Improved characteristic comparison focusing on Parameter Name
            char_exists = False
            for existing_char in detailed_SPN_dict[spn]:
                # Compare Parameter Names with more flexibility
                existing_param = existing_char.get('Parameter_Name', '').lower()
                current_param = characteristics['Parameter_Name'].lower()
                
                param_match = False
                if existing_param and current_param:
                    # Check if parameter names are similar
                    param_match = (
                        existing_param == current_param or
                        existing_param in current_param or
                        current_param in existing_param or
                        any(word in existing_param.split() for word in current_param.split())
                    )
                
                # If parameter names match or are similar, check other characteristics
                if param_match:
                    # Count matching non-empty characteristics
                    matches = sum(1 for k, v in characteristics.items() 
                                if k not in ['Platform', 'Parameter_Name'] and v != '' and 
                                existing_char.get(k, '') == v)
                    
                    # More lenient threshold for matching
                    if matches >= (non_empty_chars * 0.5):  # 50% similarity threshold
                        existing_char['Platforms'] = sorted(set(existing_char.get('Platforms', []) + [info['display_name']]))
                        char_exists = True
                        break
                    
            if not char_exists:
                characteristics['Platforms'] = [info['display_name']]
                detailed_SPN_dict[spn].append(characteristics)

    except Exception as e:
        print(f"Error processing {info['display_name']}: {str(e)}")

# Create detailed DataFrame with non-empty rows only
detailed_rows = []
for spn, variations in detailed_SPN_dict.items():
    for var in variations:
        # Only include rows that have at least some meaningful characteristics
        if any(var[k] != '' for k in ['Parameter_Name', 'Display_Type', 'Log_Type', 'Event_Codes']):
            row = {
                'SPN': spn,
                'Platforms': ', '.join(var['Platforms']),
                'Platform_Count': len(var['Platforms']),
                'Parameter_Name': var['Parameter_Name'] or 'N/A',
                'Display_Type': var['Display_Type'] or 'N/A',
                'Log_Type': var['Log_Type'] or 'N/A',
                'Event_Codes': var['Event_Codes'] or 'N/A',
                'Time_Dependency': var['Time_Dependency'] or 'N/A',
                'Event_Dependency': var['Event_Dependency'] or 'N/A',
                'Additional_Dependencies': var['Additional_Dependencies'] or 'N/A',
                'Auto_Log_Power': var['Auto_Log_Power'] or 'N/A',
                'Auto_Log_Time': var['Auto_Log_Time'] or 'N/A',
                'Event_Log_Changes': var['Event_Log_Changes'] or 'N/A'
            }
            detailed_rows.append(row)

# Create and sort the detailed DataFrame
detailed_df = pd.DataFrame(detailed_rows)
detailed_df = detailed_df.sort_values(['SPN', 'Platform_Count', 'Platforms'], ascending=[True, False, True])

# Save to CSV with timestamp
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
output_filename = os.path.join("Outputs", f"SPN_Detailed_Analysis_{timestamp}.csv")

# Save to CSV
detailed_df.to_csv(output_filename, index=False)
print(f"\nDetailed analysis saved to {output_filename}")
print(f"Found {len(detailed_SPN_dict)} unique SPNs with {len(detailed_rows)} different characteristic combinations")

0xF812
0x00F0
 - invalid literal for int() with base 16: 'f8120x00f0'
0x0206 - invalid literal for int() with base 16: '004b0x0206'

Detailed analysis saved to Outputs\SPN_Detailed_Analysis_20250804_174203.csv
Found 761 unique SPNs with 424 different characteristic combinations


In [3]:
def clean_spn_data(df, spn_column):
    """Clean and validate SPN data from a single dataframe"""
    # Remove completely empty rows
    df = df.dropna(how='all')
    
    # Clean SPN column
    def clean_spn(x):
        if pd.isna(x):
            return None
        
        # Convert to string and clean
        x = str(x).strip().lower()
        
        try:
            # If it starts with '0x', parse as hex
            if x.startswith('0x'):
                return f"0x{int(x, 16):04x}"
            # If it's a number, assume it's hex
            elif x.isdigit():
                return f"0x{int(x):04x}"
            # Otherwise keep original value
            return x
        except ValueError:
            return x  # Keep original value if conversion fails
    
    df[spn_column] = df[spn_column].apply(clean_spn)
    return df

def validate_platform_data(platform_key):
    """Validate and analyze data for a single platform"""
    info = file_mapping[platform_key]
    try:
        df = pd.read_excel(info["path"], sheet_name=info["sheet"])
        
        # Find SPN column
        spn_column = None
        for col in ['SPN', 'Parameter ID', 'Parameter ID (new)']:
            if col in df.columns:
                spn_column = col
                break
        
        if spn_column is None:
            return None, f"No SPN column found in {info['display_name']}"
        
        # Clean the data
        df = clean_spn_data(df, spn_column)
        
        # Basic validation
        validation_results = {
            'platform': info['display_name'],
            'total_rows': len(df),
            'null_spns': df[spn_column].isna().sum(),
            'unique_spns': len(df[spn_column].dropna().unique()),
            'duplicate_spns': df[spn_column].duplicated().sum(),
            'sample_spns': df[spn_column].head().tolist()
        }
        
        # Show detailed duplicate information
        show_duplicate_spns(df, spn_column, info['display_name'])
        
        return df, validation_results
        
    except Exception as e:
        return None, f"Error processing {info['display_name']}: {str(e)}"

def show_duplicate_spns(df, spn_column, platform_name):
    """Show detailed information about duplicate SPNs"""
    # Find duplicated SPNs
    duplicates = df[df[spn_column].duplicated(keep=False)]
    
    if len(duplicates) > 0:
        print(f"\nDuplicate SPNs found in {platform_name}:")
        # Group by SPN and show related rows
        for spn in duplicates[spn_column].unique():
            spn_rows = df[df[spn_column] == spn]
            print(f"\nSPN: {spn} appears {len(spn_rows)} times:")
            for idx, row in spn_rows.iterrows():
                print(f"  - Parameter Name: {row.get('Parameter Name (as read in SW data log)', 'N/A')}")
                print(f"    Display Type: {row.get('Display Type(s)', 'N/A')}")
                print(f"    Log Type: {row.get('Log Type', 'N/A')}")
                print("    ---")
    else:
        print(f"\nNo duplicate SPNs found in {platform_name}")
# Test the validation function
for platform in file_mapping.keys():
    df, results = validate_platform_data(platform)
    if isinstance(results, dict):
        print(f"\nValidation Results for {results['platform']}:")
        print(f"Total Rows: {results['total_rows']}")
        print(f"Null SPNs: {results['null_spns']}")
        print(f"Unique SPNs: {results['unique_spns']}")
        print(f"Duplicate SPNs: {results['duplicate_spns']}")
        print("Sample SPNs:", results['sample_spns'])
    else:
        print(f"\nError: {results}")


No duplicate SPNs found in AMT

Validation Results for AMT:
Total Rows: 174
Null SPNs: 0
Unique SPNs: 174
Duplicate SPNs: 0
Sample SPNs: ['0x0109', '0x010a', '0x010f', '0x010c', '0x010d']

Duplicate SPNs found in Navigator:

SPN: 0x0094 appears 2 times:
  - Parameter Name: N/A
    Display Type: N/A
    Log Type: N/A
    ---
  - Parameter Name: N/A
    Display Type: N/A
    Log Type: N/A
    ---

SPN: 0x0093 appears 2 times:
  - Parameter Name: N/A
    Display Type: N/A
    Log Type: N/A
    ---
  - Parameter Name: N/A
    Display Type: N/A
    Log Type: N/A
    ---

Validation Results for Navigator:
Total Rows: 107
Null SPNs: 0
Unique SPNs: 105
Duplicate SPNs: 2
Sample SPNs: ['????', '0x04d2', '0x048f', '0x0098', '0x010f']

No duplicate SPNs found in APU

Validation Results for APU:
Total Rows: 98
Null SPNs: 0
Unique SPNs: 98
Duplicate SPNs: 0
Sample SPNs: ['0xf00d', '0x1078', '0x1016', '0x1017', '0x1064']

Duplicate SPNs found in DMT:

SPN: new appears 6 times:
  - Parameter Name: N/